In [1]:
"""
Task 8 — Web Scraper
    ● Build a scraper using requests and BeautifulSoup.
        Create a Python pipeline that:
            1. Searches Twitter/X posts using keywords and context.
            2. Extracts:
                ○ tweet text
                ○ username
                ○ tweet link
                ○ video URL or media link (try to download video if not then just keep
                link/URL)
                ○ timestamp
                ○ engagement statistics (likes/reposts if available)
            3. Stores the collected data in:
                ○ CSV format
                ○ JSON format
        Example search contexts:
            ● AI generated videos, Robotics demos, Self-driving cars, Sports highlights
            Requirements:
            ● Use modular code
            ● Handle request failures and exceptions properly
            ● Add retry mechanisms where needed
            ● Maintain clean folder organization
"""
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json


def collect_posts(topic):
    """
    Searches DuckDuckGo for Twitter/X links related to a topic
    and returns the extracted results as a DataFrame.
    """

    url = f"https://html.duckduckgo.com/html/?q=site:twitter.com+{topic}"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    extracted_data = []

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:

            soup = BeautifulSoup(response.text, "html.parser")
            results = soup.find_all("div", class_="result")

            for item in results:

                title = item.find("a", class_="result__a")
                snippet = item.find("a", class_="result__snippet")

                if title and snippet:

                    post_link = title.get("href", "")

                    # keeping only twitter links
                    if "twitter.com" in post_link.lower():

                        extracted_data.append({
                            "post_title": title.text.strip(),
                            "post_description": snippet.text.strip(),
                            "post_url": post_link
                        })

        else:
            print("Failed to fetch search results")

    except Exception as error:
        print("Error:", error)

    return pd.DataFrame(extracted_data)



search_keyword = "Robots"

twitter_data = collect_posts(search_keyword)

if not twitter_data.empty:

    print(twitter_data.head())

    # storing data in csv format
    twitter_data.to_csv("twitter_posts.csv", index=False)

    # storing data in json format
    twitter_data.to_json("twitter_posts.json", orient="records", indent=4)

    print("\nData saved as CSV and JSON files")

else:
    print("No matching posts found")

                                          post_title  \
0                         Twitter Developer Platform   
1           Twitter. It's what's happening / Twitter   
2                                       Twitter Blog   
3                                            Twitter   
4  Lex | HypnoKink Content Creator on Twitter: "N...   

                                    post_description  \
0  User-agent: DirBuster-0.12 Disallow: / Sitemap...   
1                    Gli ultimi tweet di @robots.txt   
2  User-agent: DirBuster-0.12 Disallow: / Sitemap...   
3  User-agent: DirBuster-0.12 Disallow: / Sitemap...   
4  "Natalia's #1 in the ROBOTS category on Clips4...   

                                            post_url  
0  //duckduckgo.com/l/?uddg=https%3A%2F%2Fdev.twi...  
1  //duckduckgo.com/l/?uddg=https%3A%2F%2Ftwitter...  
2  //duckduckgo.com/l/?uddg=https%3A%2F%2Fblog.tw...  
3  //duckduckgo.com/l/?uddg=https%3A%2F%2Fabout.t...  
4  //duckduckgo.com/l/?uddg=https%3A%2F%2Ftwitter..